# MIND-large: tải từ Hugging Face + news recommendation

Dataset: [Recommenders/MIND](https://huggingface.co/datasets/Recommenders/MIND) (Microsoft News Dataset, bản large).
Trên Kaggle cần bật **Settings → Internet → On**.

Pipeline: tải & giải nén → đọc `news.tsv`/`behaviors.tsv` → baseline Popularity → Content-based (TF-IDF) → đánh giá AUC / MRR / nDCG trên tập dev.

In [ ]:
!pip -q install -U huggingface_hub

In [ ]:
import os, re, zipfile, glob
from huggingface_hub import list_repo_files, hf_hub_download

REPO = "Recommenders/MIND"
OUT = "/kaggle/working/mind" if os.path.exists("/kaggle") else "./mind"
os.makedirs(OUT, exist_ok=True)

files = list_repo_files(REPO, repo_type="dataset")
print(*files, sep="\n")   # xem repo thực tế có những file nào

In [ ]:
# Chỉ lấy các file của bản LARGE (train/dev). Bỏ qua small/demo.
large = [f for f in files if "large" in f.lower() and f.lower().endswith((".zip", ".tsv"))]
# MINDlarge_test không có nhãn click -> bỏ để đỡ tốn dung lượng
large = [f for f in large if "test" not in f.lower()]
print("Sẽ tải:", large)

for f in large:
    p = hf_hub_download(REPO, f, repo_type="dataset", local_dir=OUT)
    if p.endswith(".zip"):
        d = os.path.join(OUT, os.path.basename(p)[:-4])
        with zipfile.ZipFile(p) as z: z.extractall(d)
        print("unzipped ->", d)

In [ ]:
def find(split, name):
    hits = glob.glob(f"{OUT}/**/*{split}*/**/{name}", recursive=True) or glob.glob(f"{OUT}/**/{name}", recursive=True)
    hits = [h for h in hits if split in h.lower()]
    assert hits, f"không thấy {name} của {split}"
    return hits[0]

import pandas as pd, numpy as np
NEWS_COLS = ["news_id","category","subcategory","title","abstract","url","title_entities","abstract_entities"]
BEH_COLS  = ["impression_id","user_id","time","history","impressions"]

news = pd.concat([pd.read_csv(find(s,"news.tsv"), sep="\t", header=None, names=NEWS_COLS, usecols=range(8))
                  for s in ("train","dev")]).drop_duplicates("news_id").set_index("news_id")
train = pd.read_csv(find("train","behaviors.tsv"), sep="\t", header=None, names=BEH_COLS)
dev   = pd.read_csv(find("dev","behaviors.tsv"),   sep="\t", header=None, names=BEH_COLS)
print(news.shape, train.shape, dev.shape)
news.head(3)

## Baseline 1: Popularity (đếm số click ở tập train)

In [ ]:
from collections import Counter
clicks = Counter(re.findall(r"(N\d+)-1", " ".join(train["impressions"].dropna())))
print(clicks.most_common(5))

## Baseline 2: Content-based (TF-IDF title+abstract, user profile = trung bình lịch sử đọc)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

text = (news["category"].fillna("") + " " + news["title"].fillna("") + " " + news["abstract"].fillna(""))
tfidf = TfidfVectorizer(max_features=50000, stop_words="english", sublinear_tf=True)
X = normalize(tfidf.fit_transform(text))          # sparse, L2-norm
idx = {nid:i for i,nid in enumerate(news.index)}
print(X.shape)

In [ ]:
from sklearn.metrics import roc_auc_score

def mrr(y, s):
    o = np.argsort(s)[::-1]; r = np.asarray(y)[o]
    return (r / (np.arange(len(r)) + 1)).sum() / max(r.sum(), 1)
def ndcg(y, s, k):
    o = np.argsort(s)[::-1][:k]; y = np.asarray(y)
    dcg = (y[o] / np.log2(np.arange(len(o)) + 2)).sum()
    ideal = (np.sort(y)[::-1][:k] / np.log2(np.arange(min(k,len(y))) + 2)).sum()
    return dcg / ideal if ideal > 0 else 0.0

def user_vec(history):
    ids = [idx[h] for h in str(history).split() if h in idx][-50:]
    if not ids: return None
    v = np.asarray(X[ids].mean(axis=0)).ravel()
    n = np.linalg.norm(v); return v / n if n else None

def evaluate(df, scorer, n=20000, seed=0):
    df = df.sample(min(n, len(df)), random_state=seed)
    res = []
    for hist, imps in zip(df["history"], df["impressions"]):
        pairs = [p.rsplit("-",1) for p in imps.split()]
        ids, y = [p[0] for p in pairs], np.array([int(p[1]) for p in pairs])
        if y.sum() == 0 or y.sum() == len(y): continue
        s = np.asarray(scorer(hist, ids), dtype=float)
        res.append((roc_auc_score(y, s), mrr(y, s), ndcg(y, s, 5), ndcg(y, s, 10)))
    return dict(zip(["AUC","MRR","nDCG@5","nDCG@10"], np.mean(res, axis=0).round(4)))

def pop_scorer(hist, ids):  return [clicks.get(i, 0) for i in ids]
def cb_scorer(hist, ids):
    u = user_vec(hist)
    if u is None: return pop_scorer(hist, ids)
    rows = [idx[i] for i in ids if i in idx]
    s = np.zeros(len(ids)); m = np.array([i in idx for i in ids])
    s[m] = X[rows] @ u
    return s
def hybrid_scorer(hist, ids, a=0.7):
    p = np.log1p(pop_scorer(hist, ids)); p = p / (p.max() or 1)
    return a * np.asarray(cb_scorer(hist, ids)) + (1 - a) * p

for name, fn in [("popularity", pop_scorer), ("content-based", cb_scorer), ("hybrid", hybrid_scorer)]:
    print(name, evaluate(dev, fn))

## Gợi ý top-N cho một user
Với bản large, dev dùng để đánh giá; `MINDlarge_test` không có nhãn nên chỉ dùng khi nộp kết quả lên leaderboard.

In [ ]:
def recommend(history, k=10, candidates=None):
    cand = candidates if candidates is not None else list(news.index)
    s = hybrid_scorer(history, cand)
    top = np.argsort(s)[::-1][:k]
    return news.loc[[cand[i] for i in top], ["category","title"]]

recommend(dev["history"].iloc[0], candidates=list(news.index[:20000]))